In [1]:
import sys

sys.path.append("..")

In [2]:
from pathlib import Path

import pandas as pd
import torch

from src.models import TextEncoder
from src.preprocessing import TextProcessor
from src.utils.config import load_config
from src.utils.device import get_device

PROJECT_ROOT = Path.cwd().parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "base.yaml"

config = load_config(CONFIG_PATH)

device = get_device(
    config["training"]["device"]
)

print("Device:", device)

c:\Users\AYUSH PRASAD\Desktop\Multimodal Misinformation Detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Loading Train Data

In [3]:
train_path = (
    PROJECT_ROOT
    / config["paths"]["processed_dir"]
    / "train.csv"
)

train_df = pd.read_csv(train_path)

print("Dataset shape:", train_df.shape)

display(
    train_df[
        ["id", "text", "image_path", "label"]
    ].head()
)

Dataset shape: (12000, 9)


,id,text,image_path,label
0,4jj0fh,funny dog names you havent thought of,images/train/4jj0fh.jpg,0
1,7tykik,destyler numb,images/train/7tykik.jpg,0
2,1jfqnp,us student abandoned in san diego jail gets m ...,images/train/1jfqnp.jpg,1
3,boa5ua,this horrified styrofoam,images/train/boa5ua.jpg,0
4,1aco8p,je crois i believe canadian wwii poster x,images/train/1aco8p.jpg,0


## Initializing Tokenizer

In [4]:
text_config = config["models"]["text"]

processor = TextProcessor(
    model_name=text_config["name"],
    max_length=text_config["max_length"],
)

print("Tokenizer:", text_config["name"])
print("Max length:", text_config["max_length"])

Tokenizer: distilbert-base-uncased
Max length: 128


## Testing Tokenization

In [5]:
sample_texts = (
    train_df["text"]
    .head(4)
    .tolist()
)

encoded = processor(sample_texts)

for key, value in encoded.items():
    print(
        f"{key:15s} "
        f"shape={tuple(value.shape)} "
        f"dtype={value.dtype}"
    )

input_ids       shape=(4, 13) dtype=torch.int64
attention_mask  shape=(4, 13) dtype=torch.int64


## Initializing DistilBERT

In [6]:
encoder = TextEncoder(
    model_name=text_config["name"],
    freeze=text_config["freeze"],
)

encoder = encoder.to(device)

print("Embedding dimension:", encoder.embedding_dim)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3846.04it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 768


## Move tokenized inputs to GPU

In [7]:
encoded = {
    key: value.to(device)
    for key, value in encoded.items()
}

In [8]:
with torch.inference_mode():
    embeddings = encoder(**encoded)

print("Embedding shape:", tuple(embeddings.shape))
print("Embedding dtype:", embeddings.dtype)
print("Device:", embeddings.device)

Embedding shape: (4, 768)
Embedding dtype: torch.float32
Device: cuda:0


## Sanity Checks

In [9]:
# Verify that DistilBERT is actually frozen

trainable_parameters = sum(
    parameter.numel()
    for parameter in encoder.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in encoder.parameters()
)

print("Total parameters:     ", f"{total_parameters:,}")
print("Trainable parameters: ", f"{trainable_parameters:,}")

Total parameters:      66,362,880
Trainable parameters:  0


In [10]:
assert embeddings.ndim == 2
assert embeddings.shape[0] == len(sample_texts)
assert embeddings.shape[1] == encoder.embedding_dim
assert trainable_parameters == 0

print("Text pipeline sanity checks: PASSED")

Text pipeline sanity checks: PASSED


```
train.csv
   │
   └── text
        │
        ▼
TextProcessor
        │
        ▼
DistilBERT tokenizer
        │
        ├── input_ids
        └── attention_mask
                │
                ▼
          TextEncoder
          DistilBERT
                │
                ▼
        768-D text embedding
```